<a href="https://colab.research.google.com/github/weagan/SSM-and-Mamba/blob/main/mamba_tiny_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run PeaBrane/mamba-tiny on Colab (CPU / 1 GPU / 2 GPUs)

In [ ]:
!git clone https://github.com/PeaBrane/mamba-tiny.git
%cd mamba-tiny

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

In [ ]:
import sys
sys.path.append('/content/mamba-tiny')
from model import Mamba

In [ ]:
from transformers import AutoTokenizer
import torch

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model and move to device
model = Mamba.from_pretrained('state-spaces/mamba-370m')
model = model.to(device)
## Optional Multi-GPU
if torch.cuda.device_count() > 1:
    import torch.nn as nn
    model = nn.DataParallel(model)
    print('Using DataParallel')
model.eval()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/gpt-neox-20b')
tokenizer.pad_token = tokenizer.eos_token

# Prepare input
prompt = 'Mamba is the'
input_ids = tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)

# Generate tokens manually
max_length = 50
generated = input_ids

with torch.no_grad():
    for _ in range(max_length - input_ids.shape[1]):
        # Forward pass
        logits = model(generated)

        # Get next token (greedy decoding)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)

        # Append to generated sequence
        generated = torch.cat([generated, next_token], dim=1)

        # Stop if EOS token is generated
        if next_token.item() == tokenizer.eos_token_id:
            break

print(tokenizer.decode(generated[0], skip_special_tokens=True))